# 27 Tagged model: predictions with other efficiency targets (A100)

The tagged model was trained with each target's efficiency in the prompt (log2 speed-up over gold, from -3.0 to +3.0) and has so far been asked for +0.7. Here the kept tagged model (epoch 1 or 2, chosen in `25_sft_evaluate`) answers validation, dev and mini-dev with the targets +0.0, +1.0, +1.5, +2.0 and +3.0, each into its own run `<model>_tag<target>`. `28_sft_tag_choice` (CPU) then scores validation and picks the target by the same rule as the methods; dev and mini-dev are scored only for the chosen target.

**Runtime:** A100. About 10 minutes (one model load, under a minute per split and target).

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
EXPORT = "sft_v1"     # training data export (made once by 20_sft_pilot)
R = f"{DRIVE_ROOT}/effsql/runs"
import os
assert os.path.exists(f"{R}/{EXPORT}/export_meta.json"), "run 20_sft_pilot first: it creates the export"
import json
KEPT = json.load(open(f"{R}/{EXPORT}_epoch_choice/selection.json"))["winner"]  # epoch 1 or 2, chosen in 25
assert json.load(open(f"{R}/{KEPT}/adapter_meta.json"))["arm"] == "tagged", KEPT
# Efficiency targets to try besides the default +0.7 (KEPT itself). Training tags: 73% +0.0, 2.7% >= +1.0, 1.1% at +3.0.
TAGS = ["+0.0", "+1.0", "+1.5", "+2.0", "+3.0"]
TAG_RUNS = [f"{KEPT}_tag{t}" for t in TAGS]
print("model:", KEPT)
print("target runs:", TAG_RUNS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Predict with every target (resumable)

In [ ]:
!python -m effsql.train predict --run {KEPT} --tags {",".join(TAGS)} --splits val,dev,minidev

### Done: disconnect now, then run `28_sft_tag_choice` on a CPU runtime